# Table 1: Contemporaneous Analysis - Main Results

## Purpose
This notebook trains and evaluates XGBoost regression models for **contemporaneous (current month) nowcasting** using all available real-time data including current-period variables.

## Methodology
1. **Data Loading**: Load nowcasting dataset with current-month features (no time lags)
2. **5-Fold Cross-Validation**: Random stratified split to assess generalization
3. **Multi-Phase Modeling**: Train 4 XGBoost regressors per fold for phases 2-5
4. **Phase Classification**: Apply 20% threshold rule to convert probabilities to IPC phases
5. **Performance Metrics**: Average accuracy, sensitivity, precision, and R² across 5 folds

## Key Difference from Forecasting
- **Forecasting** uses 12-month lagged features only (predicts future)
- **Contemporaneous/Nowcasting** uses current-month real-time features (estimates present)

## Expected Outputs
- Mean cross-validation metrics (accuracy ~0.70, sensitivity ~0.91, precision ~0.80, R² ~0.64)
- CSV file: `r2_frame_cv.csv` with phase 3+ predictions vs. actual

## Note on Paths
**Execution note**: The released notebook uses the following package-relative input and output paths:

**Input**: `Nowcasting_Analysis_010825.csv`  
**Output**: `r2_frame_cv.csv`

Equivalent construction:
```python
import os
data_dir = os.path.join('..', '1.Source Data')
df = pd.read_csv(os.path.join(data_dir, 'Nowcasting_Analysis_010825.csv'))
# ... later in the notebook:
r2_frame_cv.to_csv(os.path.join('produced_graph', 'r2_frame_cv.csv'), index=False)
```

In [ ]:
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap
from xgboost import XGBClassifier
# import random forest regressor
from sklearn.ensemble import RandomForestRegressor
#import linear regression
from sklearn.linear_model import LinearRegression
# import tqdm
from tqdm import tqdm
import tqdm
#import r2_score
from sklearn.metrics import r2_score
#import confusion matrix
from sklearn.metrics import confusion_matrix
# import roc auc score
from sklearn.metrics import roc_auc_score
from food_crisis_functions import *
import json

# load contemporaneous_hyperparameters.json
with open('contemporaneous_hyperparameters.json') as f:
     best_params_xgb_regressor = json.load(f)

# load contemporaneous_hyperparameters_for_p3
with open('contemporaneous_hyperparameters_p3.json') as f:
     best_params_xgb_regressor_for_p3 = json.load(f)

# read csv
df = pd.read_csv(r'../1.Source Data/Nowcasting_Analysis_010825.csv')



# random split train and test
df_origin = df.copy()
y_pred_test = pd.DataFrame()
model_stats = pd.DataFrame()
#select phase1_percent is not na
df = df_origin[df_origin['phase1_percent'].notna()]
# Sort by region and date
df = df.sort_values(by=['area_id', 'date'])
#drop overall phase
df = df.drop(['overall_phase'], axis=1)
#for each region, set last observation to be test set
# create a series of new outcome, phase2_worse=phase2_percent+phase3_percent+phase4_percent+phase5_percent, phase3_worse=phase3_percent+phase4_percent+phase5_percent, phase4_worse=phase4_percent+phase5_percent, phase5_worse=phase5_percent
df['phase2_worse'] = df['phase2_percent'] + df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase3_worse'] = df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase4_worse'] = df['phase4_percent'] + df['phase5_percent']
df['phase5_worse'] = df['phase5_percent']
#drop phase2_percent, phase3_percent, phase4_percent, phase5_percent, phase1_percent
df = df.drop(['phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent', 'phase1_percent'], axis=1)
# Splitting the data, fivefold cross validation

from sklearn import model_selection
df['kfolds'] = -1

df = df.sample(frac=1).reset_index(drop=True)

kf = model_selection.KFold(n_splits=5)

#store accuracy_score, sensitivity, precision, overall_r2, subsample_r2 in results dataframe
results = pd.DataFrame(columns=['accuracy_score', 'sensitivity', 'precision', 'overall_r2', 'subsample_r2'])
 
for fold, (trn_, val_) in enumerate(kf.split(X=df)):
    df.loc[val_, 'kfolds'] = fold
    
for fold in range(5):
    # split train and test,20% test set
    train_df = df[df['kfolds'] != fold]
    test_df = df[df['kfolds'] == fold]
    y_pred_test = pd.DataFrame()
    # drop area_id and date
    train_df = train_df.drop(['area_id', 'date'], axis=1)
    test_df = test_df.drop(['area_id', 'date'], axis=1)
    for i in tqdm.tqdm(range(2, 6)):
        train_df_new = train_df.drop(['phase{}_worse'.format(j) for j in range(2, 6) if j != i], axis=1)
        test_df_new = test_df.drop(['phase{}_worse'.format(j) for j in range(2, 6) if j != i], axis=1)
    # drop rows with NaN in phase{}_percent
        train_df_new = train_df_new.dropna(subset=['phase{}_worse'.format(i)])
        test_df_new = test_df_new.dropna(subset=['phase{}_worse'.format(i)])
        # Split into features and target
        X_train = train_df_new.drop('phase{}_worse'.format(i), axis=1)
        y_train = train_df_new['phase{}_worse'.format(i)]
        X_test = test_df_new.drop('phase{}_worse'.format(i), axis=1)
        y_test = test_df_new['phase{}_worse'.format(i)]
        #finetune use gridsearch
        #model = xgb.XGBRegressor()
        #grid = GridSearchCV(model, param_grid_regressor_xgb, verbose=0, cv=3, scoring = 'r2', n_jobs = -1)
        #grid.fit(X_train, y_train)
        #best_params_xgb_regressor = grid.best_params_
        #best_params_xgb_regressor = best_params[i-2]
        # Train the model
        if i == 3:
            model = xgb.XGBRegressor(**best_params_xgb_regressor_for_p3)
        model = xgb.XGBRegressor(**best_params_xgb_regressor)
        model.fit(X_train, y_train)
        # Predictions
        y_pred = model.predict(X_test)
        # for y_pred_test, add a column to indicate the phase
        y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred)})], ignore_index=True)
    y_pred_test = convert_prob_to_phase(y_pred_test)
    y_test = y_pred_test['overall_phase']
    y_pred = y_pred_test['overall_phase_pred']
    cm = confusion_matrix(y_test, y_pred)
    #construct a finetume metric based on accuracy_score, f1_score, sensitivity, precision, overall_r2, subsample_r2
    accuracy_score_new, sensitivity, precision, overall_r2 = all_metrics(y_test, y_pred, cm, y_pred_test)
    results = pd.concat([results, pd.DataFrame({'accuracy_score': [accuracy_score_new], 'sensitivity': [sensitivity], 'precision': [precision], 'overall_r2': [overall_r2], 'subsample_r2': [0]})], ignore_index=True)


results.mean()


In [ ]:
r2_frame_cv = y_pred_test[['phase3_pred','phase3_test']]

#rename phase3_pred to phase3_pred_nc, phase3_test to phase3_test_nc
r2_frame_cv = r2_frame_cv.rename(columns={'phase3_pred': 'phase3_pred_cv', 'phase3_test': 'phase3_test_cv'})
#save to csv
r2_frame_cv.to_csv(r'produced_graph/r2_frame_cv.csv', index=False)